# Spark → Kusto — write via `KustoStreaming` (%%configure)

This notebook is **self-contained** — no Custom Environment required. It ships the shim via `%%configure` from a Lakehouse `Files/` location. Useful for one-off testing or when you don't have permission to create a Custom Environment.

Background on *why* the shim is needed: see [`../README.md`](../README.md).

**Before running:**
1. Upload `kusto-commons-io-shim-1.0.0.jar` to a Lakehouse `Files/` folder you can address via `abfss://`.
2. Edit the `spark.jars` value in the `%%configure` cell below to point at it.
3. Confirm the target Kusto table has streaming ingestion enabled.


## Session config

**Important:** `%%configure -f` re-creates the Spark session, so it must be the *first* cell that runs. If you've already run other cells, use **Run → Stop session** before re-running this one.

In [ ]:
%%configure -f
{
  "conf": {
    "spark.jars": "abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>.Lakehouse/Files/kusto-commons-io-shim-1.0.0.jar",
    "spark.driver.userClassPathFirst":   "true",
    "spark.executor.userClassPathFirst": "true"
  }
}

## Prereq KQL — run once in your Eventhouse query set

```kusto
.alter table FreezerTelemetry policy streamingingestion enable
.show table FreezerTelemetry policy streamingingestion
```


In [ ]:
// === Edit these to match your Eventhouse ===
val kustoUri  = "https://<your-cluster>.kusto.fabric.microsoft.com"
val database  = "<database>"
val tableName = "FreezerTelemetry"

val accessToken = mssparkutils.credentials.getToken(kustoUri)

println(s"Cluster:  $kustoUri")
println(s"Database: $database")
println(s"Table:    $tableName")

## Verify the shim is on the classpath

If either of these throws, the `%%configure` cell didn't take effect — restart the session and re-run it as the *first* cell.

In [ ]:
Class.forName("kusto_connector_shaded.org.apache.commons.io.IOUtils")

spark.sparkContext.listJars()
  .filter(j => j.contains("kusto") || j.contains("commons-io-shim"))
  .foreach(println)

## Read from Kusto

In [ ]:
val df = spark.read
  .format("com.microsoft.kusto.spark.datasource")
  .option("accessToken",   accessToken)
  .option("kustoCluster",  kustoUri)
  .option("kustoDatabase", database)
  .option("kustoQuery",    s"['$tableName'] | take 10")
  .load()

df.show()
println(s"Read ${df.count()} rows")

## Write to Kusto with `writeMode=KustoStreaming`

In [ ]:
df.write
  .format("com.microsoft.kusto.spark.datasource")
  .option("kustoCluster",  kustoUri)
  .option("kustoDatabase", database)
  .option("kustoTable",    tableName)
  .option("writeMode",     "KustoStreaming")
  .option("accessToken",   accessToken)
  .mode("append")
  .save()

println("✅ Write completed")

## Verify in Kusto

```kusto
FreezerTelemetry
| where ingestion_time() > ago(2m)
| count
```